<h2>AI-Powered Conversations with OpenAI's ChatGPT API Using Python</h2>

There has been a ChatGPT implementation where you can chat with ChatGPT extremely easily, so why might we be interested in an API instead?

Essentially, the API just plain gives you far more power and control to do more new and novel things with ChatGPT's responses, as well as the ability to integrate it with other applications.

In order to query this model, we will first need an API key. For this, you'll need an account and to set up billing. Typically, you will get some starting credit, but you may or may not, depending on when you sign up and try to use this API. You can create your account at https://platform.openai.com/

From there, go to the top right, click your profile, manage account, and then billing to add a payment method. From here, on the left side, choose API Keys under "user."

Create a key, and then copy the key's value, you will need this in your program. In the same directory that you're working in, create a "key.txt" file and copy and paste the key in there. Save and exit. This particular API costs $0.002, or a fifth of a penny, per 1,000 tokens at the time of my writing.

You will also need the `openai` Python package. You can install it with `pip install --upgrade openai`. The upgrade is there to ensure that you have the latest version, since the ChatGPT API is a new feature.

In [5]:
# %pip install openai==1.13.3

     |████████████████████████████████| 227 kB 6.0 MB/s eta 0:00:01
  Using cached cached_property-1.5.2-py2.py3-none-any.whl (7.6 kB)
  Using cached tqdm-4.66.5-py3-none-any.whl (78 kB)
  Using cached anyio-3.7.1-py3-none-any.whl (80 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
  Using cached distro-1.9.0-py3-none-any.whl (20 kB)
  Using cached pydantic-2.5.3-py3-none-any.whl (381 kB)
     |████████████████████████████████| 75 kB 4.0 MB/s  eta 0:00:01
     |████████████████████████████████| 2.1 MB 41.7 MB/s eta 0:00:01
     |████████████████████████████████| 74 kB 3.1 MB/s  eta 0:00:01
     |████████████████████████████████| 58 kB 6.3 MB/s  eta 0:00:01
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 4.5.0
    Uninstalling typing-extensions-4.5.0:
      Successfully uninstalled typing-extensions-4.5.0
You should consider upgrading via the '/opt/conda/baseenv/bin/python -m pip install --upgrade pip' command.
Note: you may need to

The way the ChatGPT API works is you need to query the model. Since these models often make use of chat history/context, every query needs to, or can, include a full message history context. 

Keep in mind, however that the maximum context length is 4096 tokens, so you need to stay under that. There are lots of options to work around this, the simplest being truncating earlier messages, but you can actually even use ChatGPT to help you to summarize and condense the previous message history. Maybe more on this later though. 4096 tokens is something like 20,000 characters, but it this can vary. Tokens are just words, bits of words, or combinations of words or cominations of bits of words. Every response from ChatGPT will inform you how many tokens you're using, so you can keep track.

Let's start with an example input from a user to the API:

**Step 1:** Import Required Libraries

- First, import the necessary libraries:

In [1]:
import openai
import requests
import json

**Step 2:** Set Up API Key and Endpoint

- Set up your API key and the API endpoint:

In [2]:
api_key = ''  # Replace with your actual OpenAI API key
api_url = 'https://api.openai.com/v1/chat/completions'

**Step 3:** Define the Request Headers

- You need to define the headers to include your API key:

In [3]:
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {api_key}'
}

**Step 4:** Prepare the Message

- Now, let's prepare the message you want to send to ChatGPT. In this case, we'll ask a simple question:

In [4]:
user_message = "Hello, how are you?"

messages = [
    {"role": "user", "content": user_message}
]

**Step 5:** Prepare the Data Payload

- Next, prepare the data payload that will be sent to the API:

In [5]:
data = {
    "model": "gpt-4",  # You can change this to "gpt-3.5-turbo" if needed
    "messages": messages,
    "temperature": 0.7
}

**Step 6:** Make the API Request

- Send the request to the OpenAI API and capture the response:

In [6]:
response = requests.post(api_url, headers=headers, data=json.dumps(data))

**Step 7:** Handle the Response

- Check if the request was successful and print the response from ChatGPT:

In [7]:
if response.status_code == 200:
    reply = response.json()['choices'][0]['message']['content']
    print("ChatGPT:", reply)
else:
    print(f"Error: {response.status_code}, {response.text}")

ChatGPT: As an artificial intelligence, I don't have feelings, but I'm here and ready to help you. How can I assist you today?


Great, looks like everything is working, now, let's see how we might combine this into our own application.

In [8]:
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {api_key}'
}
user_message = "Design a 20% more efficient air compressor."
messages = [{"role": "user", "content": user_message}]
data = {
    "model": "gpt-4",  
    "messages": messages,
    "temperature": 0.7
}
response = requests.post(api_url, headers=headers, data=json.dumps(data))
if response.status_code == 200:
    reply = response.json()['choices'][0]['message']['content']
    print("ChatGPT:", reply)
else:
    print(f"Error: {response.status_code}, {response.text}")

ChatGPT: Designing a more efficient air compressor involves a combination of several factors, including the type of compressor, the use of advanced materials, the design of the compressor and the use of advanced control systems. Below is a conceptual design of a 20% more efficient air compressor:

1. Type of Compressor: Use a rotary screw compressor. Rotary screw compressors are known for their efficiency because they have fewer moving parts and less friction. 

2. Material: Use lightweight and durable materials such as high-grade aluminum or carbon fiber. These materials reduce the weight of the compressor, which in turn reduces the energy required to operate it. The durability of these materials also minimizes wear and tear, extending the life of the compressor.

3. Design: Incorporate a two-stage compression system. The air is first compressed to an intermediate pressure, cooled, then compressed again to the final pressure. This reduces the amount of work required to achieve the sam

- Let's now take the code we just wrote and turn it into a function that can be reused for multiple messages. This will allow us to automate the process of sending messages to ChatGPT and receiving responses.

**First Define the Function**
- Let's create a function called chat_with_gpt that will take a user message as input and return ChatGPT's response.

In [9]:
def chat_with_gpt(user_message, model="gpt-4", temperature=0.7):
    # Set up API key and endpoint
    api_key = ''  # Replace with your actual OpenAI API key
    api_url = 'https://api.openai.com/v1/chat/completions'
    
    # Define the headers
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }
    
    # Prepare the message
    messages = [{"role": "user", "content": user_message}]
    
    # Prepare the data payload
    data = {
        "model": model,
        "messages": messages,
        "temperature": temperature
    }
    
    # Make the API request
    response = requests.post(api_url, headers=headers, data=json.dumps(data))
    
    # Handle the response
    if response.status_code == 200:
        reply = response.json()['choices'][0]['message']['content']
        return reply
    else:
        return f"Error: {response.status_code}, {response.text}"


**Use the Function**
- Now that the function is defined, you can easily use it to send messages and get responses from ChatGPT.

In [11]:
# Example usage
response = chat_with_gpt("Create a heat-resistant industrial tool prototype")
print("ChatGPT:", response)

ChatGPT: Designing a heat-resistant industrial tool prototype requires careful selection of materials and design considerations to ensure optimal functionality and durability. Here's a conceptual design for a heat-resistant industrial wrench:

Tool Name: ThermaWrench

1. Material Selection:
   The primary material used will be Inconel, a family of superalloys known for their excellent resistance to high temperatures, corrosion, and pressure. Additionally, the handle of the wrench will be coated with a heat-resistant compound, such as a silicone-based material, to provide protection against heat for the user.

2. Tool Design:
   The ThermaWrench will be a box-end wrench with a 12-point design. This allows for better contact with the fastener and reduces the risk of rounding it off due to excessive heat.

   The handle of the wrench will be ergonomically designed to maximize grip and comfort. The heat-resistant coating will not only provide thermal protection but also enhance grip.

   T

In [12]:
# Another example
response = chat_with_gpt("Write a case study on boosting production efficiency.")
print("ChatGPT:", response)

ChatGPT: Case Study: Boosting Production Efficiency with XYZ Manufacturing Company

Introduction:

XYZ Manufacturing Company is a leading industrial manufacturing firm based in California, producing automotive parts for over 20 years. In 2018, the company faced challenges in meeting the growing demand for its products due to inefficiencies in its production process. The management team decided to take a systematic approach to improve the overall efficiency and productivity to meet the increasing demand.

Problem:

The primary issue was the inefficient use of resources, including machinery, workforce, and raw materials. It led to longer production times, increased waste, higher costs, and decreased customer satisfaction due to delayed deliveries. The company’s production process was also lacking technological integration, which made it difficult to keep track of the inventory, orders, and production status.

Solution:

The first step was to conduct a comprehensive audit of the entire pr

<h3>Now we will demonstrate how to use Python to automate the process of reading text from a PDF document and then utilizing OpenAI's ChatGPT to generate a summary of that content.</h3>

- Install and Import necessary Library for pdf reader in Python

In [14]:
# %pip install PyPDF2

     |████████████████████████████████| 232 kB 6.3 MB/s eta 0:00:01
You should consider upgrading via the '/opt/conda/baseenv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


**Define the read_pdf Function**
- Next, define the read_pdf function to extract text from the PDF:

In [20]:
from PyPDF2 import PdfReader

# Function to read text from a PDF file using PdfReader
def read_pdf(file_path):
    text = ""
    pdf_reader = PdfReader(file_path)
    for page in pdf_reader.pages:
        text += page.extract_text() + "\n"
    return text

**Define the chat_with_gpt Function**
- Then, define the chat_with_gpt function to interact with ChatGPT:

In [16]:
def chat_with_gpt(prompt, model="gpt-4", temperature=0.7):
    # Set up API key and endpoint
    api_key = ''  # Replace with your actual OpenAI API key
    api_url = 'https://api.openai.com/v1/chat/completions'
    
    # Define the headers
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }
    
    messages = []
    # Add system message
    messages.append({'role': 'system', 'content': '''Your getting a large text summurize it into important keypoints with short description, don't say you can't do it or AI or you've limited knowledge untill some date. Ready'''})
    # Add user message with optional user-specific information
    user_message = {'role': 'user', 'content': prompt}
    messages.append(user_message)
    
    # Prepare the data payload
    data = {
        "model": model,
        "messages": messages,
        "temperature": temperature
    }
    
    # Make the API request
    response = requests.post(api_url, headers=headers, data=json.dumps(data))
    
    # Handle the response
    if response.status_code == 200:
        reply = response.json()['choices'][0]['message']['content']
        return reply
    else:
        return f"Error: {response.status_code}, {response.text}"


**Finally call the Function to Interact with ChatGPT**
- Now, define a function to send the extracted text to ChatGPT and get a summary:

In [22]:
text = read_pdf('drilling.pdf')
merged_text = ""
merged_text += text + "\n"
response = chat_with_gpt(merged_text)
print(response)

1. Market Analysis: Atlas Copco's new drilling equipment is targeted at companies in construction, mining, and oil & gas industries. The company will segment the market, conduct competitor analysis, and assess customer needs through surveys, interviews, and focus groups.

2. Product Development and Testing: The new equipment will undergo rigorous testing in real-world conditions and will adhere to all necessary industry standards and certifications.

3. Marketing Strategy: The equipment will be positioned as the most advanced and reliable option in the market. Promotion will be done through a multi-channel marketing campaign with high-quality marketing collateral.

4. Distribution and Logistics: The company will secure reliable suppliers and strengthen partnerships with distributors. An inventory management system will be implemented to track production, stock levels, and sales.

5. Sales Training: Comprehensive training sessions will be conducted for the sales team focusing on the new